# Capítulo 5: Análise Geográfica de Clientes

Neste notebook, vamos analisar a distribuição geográfica dos clientes e sua relação com as personas de consumo identificadas no Capítulo 3.

## 1. Preparação dos Dados

Carregamento dos dados originais de clientes (com a localização) e dos dados de personas previamente clusterizados.

In [6]:
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns

df_clientes = pd.read_csv('../data/df_clientes.csv')
df_personas = pd.read_csv('../data/df_clustered_final.csv')

print("Dados de clientes:", df_clientes.shape)
print("Dados de personas:", df_personas.shape)

Dados de clientes: (500, 6)
Dados de personas: (469, 6)


## 2. Merge & Limpeza

Unificar os dados de localização (Estado, Cidade, etc.) com os rótulos de persona através do `cliente_id`.

In [7]:
# Realizando o merge usando cliente_id
df_geo = pd.merge(df_clientes, df_personas, on='cliente_id', how='inner')

print("Dados após o merge:", df_geo.shape)
df_geo.head()

ValueError: You are trying to merge on object and int64 columns for key 'cliente_id'. If you wish to proceed you should use pd.concat

## 3. Análise por Estado

Visualização da contagem geral de clientes por estado.

In [ ]:
contagem_estado = df_geo['estado'].value_counts().reset_index()
contagem_estado.columns = ['Estado', 'Total_Clientes']

fig = px.bar(contagem_estado, x='Estado', y='Total_Clientes', 
             title='Quantidade de Clientes por Estado',
             color='Total_Clientes', color_continuous_scale='Viridis')
fig.show()

## 4. Cruzamento Persona vs Estado

Identificar onde estão concentrados os clientes de cada Persona, especialmente os "Campeões" (VIPs).

In [ ]:
persona_estado = pd.crosstab(df_geo['estado'], df_geo['Persona'])
persona_estado_pct = persona_estado.div(persona_estado.sum(axis=1), axis=0) * 100

fig2 = px.bar(persona_estado, barmode='group',
              title='Distribuição de Personas por Estado',
              labels={'value':'Quantidade', 'estado':'Estado'})
fig2.show()

# Heatmap para facilitar a visão de proporção
plt.figure(figsize=(10, 6))
sns.heatmap(persona_estado_pct, annot=True, fmt=".1f", cmap="YlGnBu")
plt.title("Proporção de Personas por Estado (%)")
plt.ylabel("Estado")
plt.xlabel("Persona")
plt.show()

## 5. Análise de Penetração Regional

Calcular o faturamento médio e total por estado cruzado com os dados monetários.

In [ ]:
fat_estado = df_geo.groupby('estado')['Monetario'].agg(['sum', 'mean']).reset_index()
fat_estado.columns = ['Estado', 'Faturamento_Total', 'Faturamento_Medio']
fat_estado = fat_estado.sort_values(by='Faturamento_Total', ascending=False)

fig3 = px.bar(fat_estado, x='Estado', y='Faturamento_Total',
              title='Faturamento Total por Estado',
              hover_data=['Faturamento_Medio'],
              color='Faturamento_Total', color_continuous_scale='Blues')
fig3.show()

## 6. Roadmap para Data Science Avançado

Para ir além desta exploração básica, os próximos passos poderiam envolver:

- **Geocodificação Avançada:** Obter Latitude e Longitude dos clientes via CEP para uma representação mais fluida em mapas de calor detalhados.
- **Algoritmo DBSCAN:** Utilizar agrupamento espacial baseado em densidade para identificar "manchas" lógicas de concentração populacional com as mesmas características de compra, não limitando-se às fronteiras estaduais.
- **Logística Direcionada:** Alocar centros de distribuição locais onde existam clusters órfãos com alto faturamento latente.
